# AIBackends - text embeddings and similarity with MiniLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-MiniLM-embeddings-similarity.ipynb)

Generate sentence embeddings with `all-MiniLM-L6-v2` through the Transformers runtime and
rank similar texts with the built-in `EmbeddingSimilarityWorkflow`. Covers
`examples/tasks/embed_text_transformers.py` and
`examples/workflows/embedding_similarity.py`, plus a tiny semantic-search example.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
%pip install -q "aibackends[transformers]>=0.8.1"

# If a later import fails with a transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [2]:
import aibackends

# aibackends accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch

    DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print("aibackends", aibackends.__version__)
print("device:", DEVICE)

aibackends 0.8.1
device: cpu


## 1. Embed a single text (`EmbedTask`)

In [3]:
from aibackends.models import MINILM_L6
from aibackends.runtimes import TRANSFORMERS
from aibackends.tasks import EmbedTask, create_task

embed_task = create_task(EmbedTask, runtime=TRANSFORMERS, model=MINILM_L6, device=DEVICE)
report = (
    "After our midnight game launch, thousands of players rushed to buy the limited Lunar "
    "Dragon skin. Some purchases were charged twice and the item never appeared."
)
vector = embed_task.run(report)
print("dimension:", len(vector))
print("preview:", [round(v, 4) for v in vector[:5]])

dimension: 384
preview: [-0.1102, 0.4376, 0.3482, 0.0671, 0.2137]


## 2. Rank similar support tickets (`EmbeddingSimilarityWorkflow`)

In [4]:
from aibackends.workflows import EmbeddingSimilarityWorkflow, create_workflow

similarity = create_workflow(
    EmbeddingSimilarityWorkflow, runtime=TRANSFORMERS, model=MINILM_L6, device=DEVICE
)
tickets = [
    "Customer says their Pro subscription was charged twice after renewal and wants a refund.",
    "User reports duplicate billing after upgrading to the annual plan.",
    "Customer cannot sign in because the one-time verification code never arrives.",
    "Account access is blocked after several reset attempts; no MFA email received.",
]
result = similarity.run(tickets)
for pair in result.ranked_pairs:
    print(f"[{pair.left_index}] vs [{pair.right_index}] => {pair.cosine_similarity:.4f}")

[0] vs [1] => 0.5226
[2] vs [3] => 0.4480
[0] vs [2] => 0.3424
[0] vs [3] => 0.2627
[1] vs [2] => 0.2596
[1] vs [3] => 0.2140


## 3. Semantic search over a small FAQ

Embeddings from `EmbedTask` are plain lists of floats, so any vector math works.

In [5]:
import math

faq = {
    "How do I request a refund?": "Refunds are processed within 5 business days.",
    "I never received my verification code": "Check spam or resend the code after 60s.",
    "Can I change my plan?": "Upgrade or downgrade any time from Billing settings.",
}
faq_vectors = {question: embed_task.run(question) for question in faq}


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


query = "I got billed twice, can I get my money back?"
query_vector = embed_task.run(query)
best = max(faq_vectors, key=lambda q: cosine(query_vector, faq_vectors[q]))
print(f"query: {query}\nbest match: {best} -> {faq[best]}")

query: I got billed twice, can I get my money back?
best match: How do I request a refund? -> Refunds are processed within 5 business days.
